<a href="https://www.kaggle.com/code/shamanthakreddymallu/s6e6-realmlp-lgbm-catb-xgb-dcn?scriptVersionId=325520616" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Imports & Load

**My model notebooks for your reference. This Notebook is the blend of these models only.**

[My LightGBM Notebook](https://www.kaggle.com/code/shamanthakreddymallu/s6e6-lightgbm-0-96594)

[My XGBoost Notebook](https://www.kaggle.com/code/shamanthakreddymallu/s6e6-xgboost-0-96586)

[My CatBoost Notebook](https://www.kaggle.com/code/shamanthakreddymallu/s6e6-catboost)

[My RealMLP Notebook](https://www.kaggle.com/code/shamanthakreddymallu/s6e6-realmlp-0-96611)

[My DCN Notebook](https://www.kaggle.com/code/shamanthakreddymallu/s6e6-dcn)

In [ ]:
# PATHS
TRAIN_PATH    = '/kaggle/input/competitions/playground-series-s6e6/train.csv'
TEST_PATH     = '/kaggle/input/competitions/playground-series-s6e6/test.csv'
ORIGINAL_PATH = '/kaggle/input/datasets/fedesoriano/stellar-classification-dataset-sdss17/star_classification.csv'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_classif
import lightgbm as lgb
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (balanced_accuracy_score, confusion_matrix,
                             classification_report)
from sklearn.calibration import calibration_curve
from itertools import product
from scipy.optimize import minimize

import warnings
warnings.filterwarnings('ignore')

train    = pd.read_csv(TRAIN_PATH)
test     = pd.read_csv(TEST_PATH)
original = pd.read_csv(ORIGINAL_PATH)

# Standardise original
original = original.rename(columns={'class': 'class'})
original['class'] = original['class'].str.upper().str.strip()

CLASS_COLORS = {'GALAXY': '#4C72B0', 'QSO': '#DD8452', 'STAR': '#55A868'}
CLASS_ORDER  = ['GALAXY', 'QSO', 'STAR']

print('Loaded.')
print(f'Train: {train.shape} | Test: {test.shape} | Original: {original.shape}')

# Clean Original Dataset

In [ ]:
SENTINEL = -9999
SENTINEL_COLS = ['u', 'g', 'z']  # confirmed from EDA (std >> normal range)

# How many rows are affected?
for col in SENTINEL_COLS:
    n = (original[col] < -100).sum()
    print(f'Original {col}: {n:,} sentinel rows ({n/len(original)*100:.1f}%)')

# Remove rows where ANY sentinel is present
mask_clean = ~((original['u'] < -100) | (original['g'] < -100) | (original['z'] < -100))
original_clean = original[mask_clean].copy()
print(f'\nOriginal after cleaning: {len(original_clean):,} rows '
      f'(removed {len(original) - len(original_clean):,})')

# Verify cleaned distributions look sane
print('\nOriginal (cleaned) numeric stats:')
print(original_clean[['u','g','r','i','z','redshift']].describe().T.round(3))

# Feature Engineering

## Feature Engineering Function

In [ ]:
# Fit empirical stellar locus before any feature engineering
# Using raw train STARs in low-redshift zone
_stars  = train[(train['class'] == 'STAR') & (train['redshift'] < 0.15)]
_ug     = _stars['u'] - _stars['g']
_gr     = _stars['g'] - _stars['r']
_coeffs = np.polyfit(_ug, _gr, deg=1)
LOCUS_A, LOCUS_B = float(_coeffs[0]), float(_coeffs[1])
print(f'Stellar locus fitted: g_r = {LOCUS_A:.4f} * u_g + {LOCUS_B:.4f}')

PHOTO_COLS = ['u', 'g', 'r', 'i', 'z']

def engineer_features(df, locus_a=LOCUS_A, locus_b=LOCUS_B):
    df = df.copy()

    # Group 1: Redshift Transforms
    df['log1p_redshift'] = np.log1p(df['redshift'].clip(lower=0))
    df['redshift_sq']    = df['redshift'] ** 2
    df['is_blueshift']   = (df['redshift'] < 0).astype(int)
    df['redshift_zone']  = pd.cut(
        df['redshift'],
        bins=[-np.inf, 0.0, 0.15, 0.50, 1.0, 1.3, np.inf],
        labels=[0, 1, 2, 3, 4, 5]
    ).astype(int)

    # Group 2: Color Indices
    df['u_g'] = df['u'] - df['g']
    df['g_r'] = df['g'] - df['r']
    df['r_i'] = df['r'] - df['i']
    df['i_z'] = df['i'] - df['z']
    df['u_r'] = df['u'] - df['r']
    df['u_z'] = df['u'] - df['z']
    df['g_i'] = df['g'] - df['i']
    df['g_z'] = df['g'] - df['z']
    df['r_z'] = df['r'] - df['z']

    # Group 3: Magnitude Summary Stats
    df['mean_mag']  = df[PHOTO_COLS].mean(axis=1)
    df['mag_range'] = df[PHOTO_COLS].max(axis=1) - df[PHOTO_COLS].min(axis=1)
    df['mag_std']   = df[PHOTO_COLS].std(axis=1)

    # Group 4: Interaction Features
    df['redshift_x_gr']     = df['redshift'] * df['g_r']
    df['redshift_x_ug']     = df['redshift'] * df['u_g']
    df['redshift_x_uz']     = df['redshift'] * df['u_z']
    df['log_redshift_x_gr'] = df['log1p_redshift'] * df['g_r']

    # Group 5: Positional — redshift-scaled Cartesian
    alpha_rad = np.radians(df['alpha'])
    delta_rad = np.radians(df['delta'])
    r_dist    = df['log1p_redshift']
    df['phys_x'] = r_dist * np.cos(delta_rad) * np.cos(alpha_rad)
    df['phys_y'] = r_dist * np.cos(delta_rad) * np.sin(alpha_rad)
    df['phys_z'] = r_dist * np.sin(delta_rad)

    # Group 6: Stellar Locus + Low-z Zone Features
    df['locus_gr_pred']  = locus_a * df['u_g'] + locus_b
    df['locus_distance'] = df['g_r'] - df['locus_gr_pred']
    df['locus_dist_abs'] = df['locus_distance'].abs()

    return df

train          = engineer_features(train)
test           = engineer_features(test)
original_clean = engineer_features(original_clean)

print('Feature engineering applied.')
print(f'Train shape: {train.shape}')

## Categorical Encoding

In [ ]:
# Ordinal encoding: spectral_type ordered by mean redshift
# Physically meaningful: lower redshift types (cooler/nearer) vs higher (hotter/farther)
st_redshift_order = (
    train.groupby('spectral_type')['redshift']
    .mean()
    .sort_values()
    .index.tolist()
)
print('spectral_type ordered by mean redshift:', st_redshift_order)

st_ordinal_map = {st: i for i, st in enumerate(st_redshift_order)}
print('Ordinal map:', st_ordinal_map)

for df in [train, test]:
    df['spectral_type_ord'] = df['spectral_type'].map(st_ordinal_map)

# Apply same map to original_clean (NaN for any unseen values)
original_clean['spectral_type_ord'] = original_clean.get(
    'spectral_type', pd.Series(dtype=str)
).map(st_ordinal_map) if 'spectral_type' in original_clean.columns else np.nan

# galaxy_population binary encoding 
gp_map = {'Red_Sequence': 1, 'Blue_Cloud': 0}

for df in [train, test]:
    df['galaxy_pop_bin'] = df['galaxy_population'].map(gp_map)

# Original doesn't have this — will be NaN (LightGBM handles natively)
original_clean['galaxy_pop_bin'] = np.nan

# Combined categorical interaction
for df in [train, test]:
    df['spec_gpop'] = df['spectral_type'] + '_' + df['galaxy_population']

# Encode combined label (fit on train, apply to test)
spec_gpop_vals = train['spec_gpop'].unique()
spec_gpop_map  = {v: i for i, v in enumerate(spec_gpop_vals)}
print('\nspec_gpop unique values and encoding:')
for k, v in sorted(spec_gpop_map.items()):
    print(f'  {k}: {v}')

for df in [train, test]:
    df['spec_gpop_enc'] = df['spec_gpop'].map(spec_gpop_map)

# Target encoding — spectral_type
# For each class, encode as P(class | spectral_type) — useful for all 3 classes
for cls in CLASS_ORDER:
    col_name = f'spec_target_{cls.lower()}'
    target_map = train.groupby('spectral_type')['class'].apply(
        lambda x: (x == cls).mean()
    )
    print(f'\nTarget encode spectral_type → P({cls}):')
    print(target_map.round(3))
    for df in [train, test]:
        df[col_name] = df['spectral_type'].map(target_map)

# Target encoding — galaxy_population
for cls in CLASS_ORDER:
    col_name = f'gpop_target_{cls.lower()}'
    target_map = train.groupby('galaxy_population')['class'].apply(
        lambda x: (x == cls).mean()
    )
    print(f'\nTarget encode galaxy_population → P({cls}):')
    print(target_map.round(3))
    for df in [train, test]:
        df[col_name] = df['galaxy_population'].map(target_map)

print('\nCategorical encoding done.')

## Integrate Original Data

In [ ]:
SHARED_NUMERIC = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']

ENGINEERED_COLS = [
    'log1p_redshift', 'redshift_sq', 'is_blueshift', 'redshift_zone',
    'u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'u_z', 'g_i', 'g_z', 'r_z',
    'mean_mag', 'mag_range', 'mag_std',
    'redshift_x_gr', 'redshift_x_ug', 'redshift_x_uz', 'log_redshift_x_gr',
    'phys_x', 'phys_y', 'phys_z',
]

CATEGORICAL_ENGINEERED = [
    'spectral_type_ord', 'galaxy_pop_bin', 'spec_gpop_enc',
    'spec_target_galaxy', 'spec_target_qso', 'spec_target_star',
    'gpop_target_galaxy', 'gpop_target_qso', 'gpop_target_star',
]

ALL_FEATURE_COLS = (
    SHARED_NUMERIC
    + ENGINEERED_COLS
    + CATEGORICAL_ENGINEERED
)

# Build original augmentation rows — set missing categorical-derived cols to NaN
orig_aug = original_clean[SHARED_NUMERIC + ['class']].copy()
orig_aug['source'] = 'original'

for col in ENGINEERED_COLS:
    if col in original_clean.columns:
        orig_aug[col] = original_clean[col].values
    else:
        orig_aug[col] = np.nan

for col in CATEGORICAL_ENGINEERED:
    orig_aug[col] = np.nan  # all NaN — no spectral_type / galaxy_population

train['source'] = 'kaggle'
train_augmented = pd.concat([train, orig_aug], ignore_index=True)

print(f'Kaggle train only : {len(train):,}')
print(f'Original (cleaned): {len(orig_aug):,}')
print(f'Augmented train   : {len(train_augmented):,}')
print(f'\nClass distribution in augmented:')
print(train_augmented['class'].value_counts())

## Final Features

In [ ]:
FINAL_FEATURES = [
    'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift',
    'log1p_redshift', 'redshift_sq', 'redshift_zone',
    'u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'u_z', 'g_i', 'g_z', 'r_z',
    'mean_mag', 'mag_range', 'mag_std',
    'redshift_x_gr', 'redshift_x_ug', 'redshift_x_uz', 'log_redshift_x_gr',
    'phys_x', 'phys_y', 'phys_z',
    'spectral_type_ord', 'galaxy_pop_bin', 'spec_gpop_enc',
    'spec_target_galaxy', 'spec_target_qso', 'spec_target_star',
    'gpop_target_galaxy', 'gpop_target_qso', 'gpop_target_star',
    'locus_distance', 'locus_dist_abs',
]
print(f'FINAL_FEATURES: {len(FINAL_FEATURES)} features')

## Sanity Checks

In [ ]:
# Check for any unexpected inf/nan in final features on kaggle train
print('NaN/Inf check — Kaggle Train')
for col in FINAL_FEATURES:
    n_nan = train[col].isna().sum()
    n_inf = np.isinf(train[col].replace([np.inf, -np.inf], np.nan).fillna(0)).sum()
    if n_nan > 0 or n_inf > 0:
        print(f'  {col}: {n_nan} NaN, {n_inf} Inf')
print('  (no output = all clean)')

print('\n NaN/Inf check — Test')
for col in FINAL_FEATURES:
    n_nan = test[col].isna().sum()
    n_inf = np.isinf(test[col].replace([np.inf, -np.inf], np.nan).fillna(0)).sum()
    if n_nan > 0 or n_inf > 0:
        print(f'  {col}: {n_nan} NaN, {n_inf} Inf')
print('  (no output = all clean)')

print(f'\nFinal shapes:')
print(f'  train[FINAL_FEATURES]: {train[FINAL_FEATURES].shape}')
print(f'  test[FINAL_FEATURES] : {test[FINAL_FEATURES].shape}')
print(f'  train_augmented      : {train_augmented.shape}')

# Modeling

In [ ]:
le = LabelEncoder()
y  = le.fit_transform(train['class'])

N_FOLDS = 5
SEED = 42
N_CLASSES = 3

print(f'Class mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# Training

In [ ]:
y_aug   = le.transform(train_augmented['class'])
kag_idx = np.where(train_augmented['source'] == 'kaggle')[0]
y_eval  = y_aug[kag_idx]


oof_lgb  = np.load(f'/kaggle/input/notebooks/shamanthakreddymallu/s6e6-lightgbm-0-96594/oof_lgb.npy')[kag_idx]
oof_mlp  = np.load(f'/kaggle/input/notebooks/shamanthakreddymallu/s6e6-realmlp-0-96611/oof_mlp.npy')[kag_idx]
oof_dcn  = np.load(f'/kaggle/input/notebooks/shamanthakreddymallu/s6e6-dcn/oof_dcn.npy')[kag_idx]
oof_xgbcd  = np.load(f'/kaggle/input/notebooks/cdeotte/xgb-v5-for-s6e6/train_oof/xgb-5_oof.npy')[kag_idx]
oof_catcd  = np.load(f'/kaggle/input/notebooks/cdeotte/cat-v3-for-s6e6/train_oof/cat-3_oof.npy')[kag_idx]
oof_mlpcd  = np.load(f'/kaggle/input/notebooks/cdeotte/realmlp-v5-for-s6e6/train_oof/realmlp-5_oof.npy')[kag_idx]

test_lgb = np.load(f'/kaggle/input/notebooks/shamanthakreddymallu/s6e6-lightgbm-0-96594/test_lgb.npy')
test_mlp = np.load(f'/kaggle/input/notebooks/shamanthakreddymallu/s6e6-realmlp-0-96611/test_mlp.npy')
test_dcn = np.load(f'/kaggle/input/notebooks/shamanthakreddymallu/s6e6-dcn/test_dcn.npy')
test_xgbcd = np.load(f'/kaggle/input/notebooks/cdeotte/xgb-v5-for-s6e6/test_preds/xgb-5_test_preds.npy')
test_catcd = np.load(f'/kaggle/input/notebooks/cdeotte/cat-v3-for-s6e6/test_preds/cat-3_test_preds.npy')
test_mlpcd = np.load(f'/kaggle/input/notebooks/cdeotte/realmlp-v5-for-s6e6/test_preds/realmlp-5_test_preds.npy')

MODEL_NAMES = ['LGB', 'MLP', 'DCN', 'XGB-CD', 'CAT-CD', 'MLP-CD']
oof_stack   = [oof_lgb, oof_mlp, oof_dcn, oof_xgbcd, oof_catcd, oof_mlpcd]
test_stack  = [test_lgb, test_mlp, test_dcn, test_xgbcd, test_catcd, test_mlpcd]

for name, o in zip(MODEL_NAMES, oof_stack):
    print(f'{name}: OOF BA = {balanced_accuracy_score(y_eval, o.argmax(1)):.5f}')

# Blending

In [ ]:
def blend_score(w):
    w = np.abs(w); w = w / w.sum()
    blended = sum(wi * p for wi, p in zip(w, oof_stack))
    return -balanced_accuracy_score(y_eval, blended.argmax(1))

res     = minimize(blend_score, x0=np.ones(len(oof_stack)) / len(oof_stack),
                   method='Nelder-Mead')
blend_w = np.abs(res.x) / np.abs(res.x).sum()

oof_blend  = sum(wi * p for wi, p in zip(blend_w, oof_stack))
test_blend = sum(wi * p for wi, p in zip(blend_w, test_stack))

for name, w in zip(MODEL_NAMES, blend_w):
    print(f'  {name}: {w:.3f}')
ba_blend = balanced_accuracy_score(y_eval, oof_blend.argmax(1))
ba_best  = max(balanced_accuracy_score(y_eval, o.argmax(1)) for o in oof_stack)
print(f'Best solo OOF : {ba_best:.5f}')
print(f'Blended OOF   : {ba_blend:.5f}  ({ba_blend - ba_best:+.5f})')

# Class Weights

In [ ]:
def optimize_class_weights(probs, y_true, coarse=21, refine=21):
    grid       = np.linspace(0.6, 1.6, coarse)
    best_score = balanced_accuracy_score(y_true, probs.argmax(1))
    best_w     = np.ones(3)
    for wq, ws in product(grid, grid):
        w = np.array([1.0, wq, ws])
        s = balanced_accuracy_score(y_true, (probs * w).argmax(1))
        if s > best_score:
            best_score, best_w = s, w
    fq = np.linspace(best_w[1] - 0.05, best_w[1] + 0.05, refine)
    fs = np.linspace(best_w[2] - 0.05, best_w[2] + 0.05, refine)
    for wq, ws in product(fq, fs):
        w = np.array([1.0, wq, ws])
        s = balanced_accuracy_score(y_true, (probs * w).argmax(1))
        if s > best_score:
            best_score, best_w = s, w
    return best_w, best_score

global_w, global_score = optimize_class_weights(oof_blend, y_eval)

test_final = test_blend * global_w
test_final = test_final / test_final.sum(axis=1, keepdims=True)

print(f'Class weights  : GALAXY={global_w[0]:.3f}  QSO={global_w[1]:.3f}  STAR={global_w[2]:.3f}')
print(f'OOF BA blend   : {ba_blend:.5f}')
print(f'OOF BA + weights: {global_score:.5f}')
print(f'Gain           : {global_score - ba_blend:+.5f}')

# Post Modeling Analysis

## OOF Score + Confusion Matrix

In [ ]:
y_kag     = y_eval
probs_kag = oof_blend * global_w
probs_kag = probs_kag / probs_kag.sum(axis=1, keepdims=True)
preds_kag = probs_kag.argmax(axis=1)
train_kag = train_augmented.iloc[kag_idx].reset_index(drop=True)

cm     = confusion_matrix(y_kag, preds_kag)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

print(f'OOF Balanced Accuracy (blend + weights): {global_score:.5f}')
print()
print(classification_report(y_kag, preds_kag, target_names=le.classes_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_,
            ax=axes[0], linewidths=0.5, annot_kws={'size': 12})
axes[0].set_title('Confusion Matrix — Raw Counts', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_,
            ax=axes[1], linewidths=0.5, vmin=0, vmax=100,
            annot_kws={'size': 12})
axes[1].set_title('Confusion Matrix — Recall %', fontweight='bold')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')

plt.suptitle('OOF Confusion Matrix — 4-Model Blend + Class Weights',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

for i, cls in enumerate(le.classes_):
    print(f'  {cls:8s}: {cm_pct[i,i]:.2f}%  ({cm[i,i]:,} / {cm[i].sum():,})')

## Calibration

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for cls_idx, cls_name in enumerate(le.classes_):
    ax     = axes[cls_idx]
    y_bin  = (y_kag == cls_idx).astype(int)
    y_prob = probs_kag[:, cls_idx]
    prob_true, prob_pred = calibration_curve(y_bin, y_prob, n_bins=20)
    ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Perfect')
    ax.plot(prob_pred, prob_true, 'o-',
            color=CLASS_COLORS[cls_name], lw=2, ms=5, label=cls_name)
    ax.fill_between(prob_pred, prob_true, prob_pred,
                    alpha=0.15, color=CLASS_COLORS[cls_name])
    ece = np.mean(np.abs(prob_true - prob_pred))
    ax.text(0.05, 0.91, f'ECE ≈ {ece:.4f}', transform=ax.transAxes,
            fontsize=10, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Fraction of Positives')
    ax.set_title(f'Calibration — {cls_name}', fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)

plt.suptitle('OOF Probability Calibration — Blend', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## Error Analysis

In [ ]:
error_df = train_kag[['redshift', 'g_r', 'u_g', 'spectral_type',
                       'galaxy_population', 'class']].copy()
error_df['y_true']   = le.inverse_transform(y_kag)
error_df['y_pred']   = le.inverse_transform(preds_kag)
error_df['correct']  = (error_df['y_true'] == error_df['y_pred'])
error_df['max_prob'] = probs_kag.max(axis=1)

errors  = error_df[~error_df['correct']].copy()
correct = error_df[ error_df['correct']].copy()

print(f'Total errors  : {len(errors):,}  ({len(errors)/len(train_kag)*100:.2f}%)')
print(f'Total correct : {len(correct):,} ({len(correct)/len(train_kag)*100:.2f}%)')

print('\nError breakdown by TRUE class:')
err_by_true = errors.groupby('y_true').agg(
    n_errors=('correct', 'count'),
    avg_confidence=('max_prob', 'mean'),
    median_redshift=('redshift', 'median'),
)
err_by_true['pct_of_class'] = [
    len(errors[errors['y_true'] == c]) /
    len(train_kag[train_kag['class'] == c]) * 100
    for c in err_by_true.index
]
print(err_by_true.round(3).to_string())

print('\nTop confusion pairs (true → predicted):')
print(
    errors.groupby(['y_true', 'y_pred'])
    .agg(count=('correct','count'),
         avg_confidence=('max_prob','mean'),
         median_redshift=('redshift','median'),
         median_gr=('g_r','median'))
    .sort_values('count', ascending=False)
    .to_string()
)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, (tc, pc) in zip(axes[0],
        [('GALAXY','QSO'), ('GALAXY','STAR'), ('STAR','GALAXY')]):
    mask    = (errors['y_true'] == tc) & (errors['y_pred'] == pc)
    err_sub = errors[mask]
    ax.hist(train_kag[train_kag['class']==tc]['redshift'], bins=80, alpha=0.3,
            color=CLASS_COLORS[tc], density=True, label=f'All {tc}')
    ax.hist(err_sub['redshift'], bins=80, alpha=0.7,
            color='#C44E52', density=True, label=f'→ {pc} ({len(err_sub):,})')
    ax.set_title(f'True {tc} → Pred {pc}', fontweight='bold')
    ax.set_xlabel('redshift'); ax.set_ylabel('Density')
    ax.legend(fontsize=8)

for ax, cls in zip(axes[1], CLASS_ORDER):
    mask     = error_df['y_true'] == cls
    err_conf = error_df.loc[mask & ~error_df['correct'], 'max_prob']
    ok_conf  = error_df.loc[mask &  error_df['correct'], 'max_prob']
    ax.hist(ok_conf,  bins=60, alpha=0.5, color=CLASS_COLORS[cls],
            density=True, label=f'Correct ({len(ok_conf):,})')
    ax.hist(err_conf, bins=60, alpha=0.6, color='#C44E52',
            density=True, label=f'Error ({len(err_conf):,})')
    ax.axvline(err_conf.median(), color='red', linestyle='--', lw=1.5,
               label=f'Error median={err_conf.median():.2f}')
    ax.set_title(f'{cls} — Confidence', fontweight='bold')
    ax.set_xlabel('Max Predicted Probability')
    ax.legend(fontsize=8)

plt.suptitle('Error Analysis — 4-Model Blend',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Submission

In [ ]:
labels = le.inverse_transform(test_final.argmax(axis=1))

sub = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv')
sub['class'] = labels
sub.to_csv('submission.csv', index=False)

low_conf = (test_final.max(axis=1) < 0.6).sum()
print(f'Saved: submission.csv')
print(pd.Series(labels).value_counts().to_string())
print(f'Low-confidence (<60%): {low_conf:,} ({low_conf/len(labels)*100:.2f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(test_final.max(axis=1), bins=100,
             color='#4C72B0', edgecolor='white', alpha=0.85)
axes[0].axvline(0.6, color='red',    linestyle='--', label='60%')
axes[0].axvline(0.9, color='orange', linestyle='--', label='90%')
axes[0].set_xlabel('Max Probability'); axes[0].set_ylabel('Count')
axes[0].set_title('Confidence Distribution', fontweight='bold')
axes[0].legend()

for idx, cls_name in enumerate(le.classes_):
    mask = test_final.argmax(axis=1) == idx
    axes[1].hist(test_final[mask].max(axis=1), bins=60, alpha=0.6,
                 color=CLASS_COLORS[cls_name], density=True, label=cls_name)
axes[1].set_xlabel('Max Probability'); axes[1].set_ylabel('Density')
axes[1].set_title('Confidence by Class', fontweight='bold')
axes[1].legend()
plt.suptitle('Test Prediction Confidence — 4-Model Blend', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'\nBlend weights         : ' +
      '  '.join(f'{n}={w:.3f}' for n, w in zip(MODEL_NAMES, blend_w)))
print(f'OOF BA blend          : {ba_blend:.5f}')
print(f'OOF BA blend + weights: {global_score:.5f}')